In [1]:
import pandas as pd
import seaborn as sns
import sys
sys.path.insert(0, "/Users/joshua/Developer/civetqc")

In [2]:
from civetqc.dataset import Dataset
from civetqc.model import BaseModel
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline, make_pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, fbeta_score, make_scorer
from sklearn.neighbors import KNeighborsClassifier, NeighborhoodComponentsAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, QuantileTransformer
from sklearn.svm import SVC

In [ ]:
data = Dataset()

In [ ]:
model = BaseModel(data)

In [ ]:
model.predicted

In [ ]:
pipeline = make_pipeline(
    StandardScaler(),
    SMOTE(random_state=0),
    NeighborhoodComponentsAnalysis(),
    KNeighborsClassifier()
)

params = [{
    'kneighborsclassifier__n_neighbors': list(range(1, 11)),
    'kneighborsclassifier__leaf_size': [10, 30, 50]
}]

gs_knn = GridSearchCV(
    pipeline,
    param_grid=params,
    scoring=f2_scorer,
    cv=5
)

In [ ]:
gs_knn.fit(data.train['Features'], data.train['Target'])
gs_knn.best_params_

In [ ]:
pred = gs_knn.predict(data.test['Features'])
print(f"f2 score: {gs_knn.score(data.test['Features'], data.test['Target'])}")
print(classification_report(data.test['Target'], pred))

In [ ]:
svm_pipe = Pipeline([
    ('mms', QuantileTransformer()),
    ('nca', NeighborhoodComponentsAnalysis()),
    ('svc', SVC(class_weight="balanced")),
])

In [ ]:
# svm params
svm_params = [{
    'svc__C': [0.1, 1, 10, 100, 1000],
    'svc__gamma': [1, 0.1, 0.01, 0.001, 0.0001],
    'svc__kernel': ['rbf', 'poly', 'sigmoid']
}]

In [ ]:
gs_svm = GridSearchCV(
    svm_pipe,
    param_grid=svm_params,
    cv=5,
    refit=True,
    scoring=f2_scorer
)

In [ ]:
gs_svm.fit(data.train['Features'], data.train['Target'])
gs_svm.best_params_

In [ ]:
pred = gs_svm.predict(data.test['Features'])
print(f"f2 score: {gs_svm.score(data.test['Features'], data.test['Target'])}")
print(classification_report(data.test['Target'], pred))

In [ ]:
rf_pipe = Pipeline([
    ('mms', StandardScaler()),
    ('rf', RandomForestClassifier()),
])

In [ ]:
rf_params = {
    'rf__max_depth': [2, 4, 6, 8],
    'rf__class_weight': [{0: 1, 1: 1}, {0: 2, 1: 1}, {0: 3, 1: 1}, {0: 10, 1: 1}, {0: 1, 1: 2}]
}

In [ ]:
gs_rf = GridSearchCV(
    rf_pipe,
    param_grid=rf_params,
    cv=5,
    scoring=f2_scorer
)

In [ ]:
gs_rf.fit(data.train['Features'], data.train['Target'])
gs_rf.best_params_

In [ ]:
pred = gs_rf.predict(data.test['Features'])
print(f"f2 score: {gs_rf.score(data.test['Features'], data.test['Target'])}")
print(classification_report(data.test['Target'], pred))

In [ ]:
nn_pipe = Pipeline([
    ('std', StandardScaler()),
    ('mlp', MLPClassifier(max_iter=1000)),
])

parameter_space = {
    'mlp__hidden_layer_sizes': [(50,50,50), (50,100,50), (100,)],
    'mlp__activation': ['tanh', 'relu'],
    'mlp__solver': ['sgd', 'adam'],
    'mlp__alpha': [0.0001, 0.05],
    'mlp__learning_rate': ['constant','adaptive'],
}

gs_nn = GridSearchCV(
    nn_pipe,
    param_grid=parameter_space,
    n_jobs=-1,
    cv=5,
    scoring=f2_scorer
)

gs_nn.fit(data.train['Features'], data.train['Target'])
gs_nn.best_params_

pred = gs_nn.predict(data.test['Features'])
print(f"f2 score: {gs_nn.score(data.test['Features'], data.test['Target'])}")
print(classification_report(data.test['Target'], pred))